In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from pathlib import Path

PRED_PATH = Path("../../../data/analysis/Fig5/predictions_N_ensemble_10seed_sorted_centered_err.json")
MIN_RESIDUES = 5
FIG_DIR = Path("Figs")
FIG_DIR.mkdir(exist_ok=True)


def save(fig, name):
    fig.savefig(FIG_DIR / f"{name}.svg", bbox_inches="tight")
    fig.savefig(FIG_DIR / f"{name}.png", dpi=200, bbox_inches="tight")
    print("saved", name)

GT_PATH = ("../../../data/rBMRB/cluster_30p/PANAV/test.json")
gt_json = json.load(open(GT_PATH))
inv = {v: k for k, v in gt_json["entry_ID"].items()}

PRED_PATH_UBQ = "../../../data/analysis/Fig5/predictions_N_ensemble_10seed_sorted_centered_err.json"
pred_ubq = json.load(open(PRED_PATH_UBQ))

def _mean_gt(pid):
    p = np.array(pred_ubq["pred"][pid], dtype=np.float64)
    mask = np.array(pred_ubq["mask"][pid], dtype=bool)
    oof_resid = np.array(pred_ubq["oof_resid"][pid], dtype=np.float64)
    oof_mask = np.array(pred_ubq["oof_mask"][pid], dtype=bool)
    n = min(len(p), len(mask), len(oof_resid), len(oof_mask))
    valid = mask[:n] & oof_mask[:n]
    g = oof_resid[:n][valid]
    return float(np.nanmean(g[np.isfinite(g)]))

REF_PID = "30693_1"   # plain WT ubiquitin, 76 aa
CORE_LEN = 76          # every sequence's last 76 residues line up 1:1 by position (point
                        # mutations only, no internal indels) -- only the N-terminus varies
                        # in length (expression-tag prefixes), so a real aligner isn't needed

DESCRIPTIONS = {
    "11505_1": "Q41N alt. structure",
    "11547_1": "UNDER-PRED, alt. state",
    "15410_1": "WT",
    "16228_1": "SAIL-labeled",
    "17439_1": "pH 7.2",
    "17769_1": "WT (reference)",
    "18611_1": "UNDER-PRED, A-state",
    "18831_1": "U14Ub1 variant",
    "19670_1": "lysine-free",
    "25972_1": "V26A",
    "27512_1": "UNDER-PRED, G76C",
    "30693_1": "RDC derived ensemble",
    "34351_1": "ensemble (CoMAND)",
    "4493_1":  "hydrophobic core variant",
    "4769_1":  "yeast, pH 7.5",
    "51725_1": "T12C",
    "52529_1": "K27M",
    "5387_1":  "reverse micelles",
    "6466_1":  "relaxation study",
}

PIDS_UNSORTED = list(DESCRIPTIONS.keys())
gt_by_pid = {pid: _mean_gt(pid) for pid in PIDS_UNSORTED}
PIDS = sorted(PIDS_UNSORTED, key=lambda p: -gt_by_pid[p])   # worst (highest err) first -> top row

seqs = {pid: "".join(gt_json["sequence"][inv[pid]]) for pid in PIDS}
ref_seq = seqs[REF_PID]
assert len(ref_seq) == CORE_LEN

prefix_len = {pid: len(seqs[pid]) - CORE_LEN for pid in PIDS}
max_prefix = max(prefix_len.values())

# every row: (max_prefix - own_prefix) leading gaps, then own prefix (if any), then the 76-aa core
aligned = {}
for pid in PIDS:
    pl = prefix_len[pid]
    aligned[pid] = "-" * (max_prefix - pl) + seqs[pid]
ref_row = "-" * max_prefix + ref_seq
aln_len = max_prefix + CORE_LEN

LABELS = {pid: f"{pid.split('_')[0]:5s}: {DESCRIPTIONS[pid]}  (err={gt_by_pid[pid]:.2f})" for pid in PIDS}

# ---- plot ----
def set_style(base=13, family="Arial"):
    plt.rcParams.update({"font.family": family, "font.size": base, "svg.fonttype": "none"})
set_style()

MATCH_C, MISMATCH_C, GAP_C, REF_C = "#EAECEE", "#E06C75", "#D5D8DC", "#61AFEF"

rows = PIDS
n_rows = len(rows)
fig, ax = plt.subplots(figsize=(max(14, aln_len * 0.16), n_rows * 0.4 + 1))

for i, pid in enumerate(rows):
    y = n_rows - i - 1
    seq_row = aligned[pid]
    for j, ch in enumerate(seq_row):
        ref_ch = ref_row[j]
        if pid == REF_PID:
            color = REF_C
        elif ch == "-":
            color = GAP_C
        elif ch == ref_ch:
            color = MATCH_C
        else:
            color = MISMATCH_C
        ax.add_patch(Rectangle((j, y), 1, 1, facecolor=color, edgecolor="white", lw=0.4))
        if ch != "-":
            ax.text(j + 0.5, y + 0.5, ch, ha="center", va="center", fontsize=9, font="Courier",
                     fontweight="bold" if color == MISMATCH_C else "normal")

ax.set_xlim(0, aln_len)
ax.set_ylim(0, n_rows)
ax.set_yticks([n_rows - i - 0.5 for i in range(n_rows)])
ax.set_yticklabels([LABELS[p] for p in rows], fontsize=9, family="Arial")
# x axis in reference-WT numbering: column max_prefix = WT residue 1, tag residues before it are -1, -2, ... (no 0)
def col_of(wt_num):
    return max_prefix + wt_num - (1 if wt_num > 0 else 0)

tick_nums = ([k for k in range(-5, -max_prefix - 1, -5)][::-1]       # -10, -5 inside the tag region
             + [1] + list(range(5, CORE_LEN + 1, 5)))                 # 1, 5, 10, ..., 75
ax.set_xticks([col_of(k) + 0.5 for k in tick_nums])                   # +0.5 -> centre of the cell
ax.set_xticklabels(tick_nums, fontsize=9, family="Arial")
ax.set_xlabel(f"residue number (WT ubiquitin, 30693)")
# ax.set_title(f"Ubiquitin sequence alignment, sorted by error, reference={REF_PID}", fontsize=12)
for s in ("top", "right", "left"):
    ax.spines[s].set_visible(False)
ax.tick_params(left=False)
plt.tight_layout()
save(fig, "ubiquitin_outliers_sequence_alignment")   # uncomment to save
plt.show()